In [1]:
import os
import copy
import numpy as np
import cv2

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from torchvision.datasets import ImageFolder
from torchvision.models import ResNet50_Weights

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
class CTDataset(Dataset):
    def __init__(self, image_paths, labels, image_size=256):
        self.image_paths = image_paths
        self.labels = labels
        self.image_size = image_size
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        label = self.labels[idx]

        img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        img = cv2.resize(img, (self.image_size, self.image_size))

        img = img.astype("float32")
        if img.max() > 255:
            img = img / 65535.0
        else:
            img = img / 255.0

        img = torch.from_numpy(img).float().unsqueeze(0)
        img = img.repeat(3, 1, 1)

        img = (img - self.mean) / self.std

        label = torch.tensor(label, dtype=torch.long)
        return img, label

In [4]:
train_dir = r"C:\Users\rgzep\Documents\IP data\data_png_wholeslice_lung256_processed\Centre10_split\train"
val_dir   = r"C:\Users\rgzep\Documents\IP data\data_png_wholeslice_lung256_processed\Centre10_split\val"
test_dir  = r"C:\Users\rgzep\Documents\IP data\data_png_wholeslice_lung256_processed\Centre10_split\test"

img_size = 256
batch_size = 16
num_workers = 0
image_size = 256

save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)

In [5]:
train_base = ImageFolder(train_dir)
val_base = ImageFolder(val_dir)
test_base = ImageFolder(test_dir)

train_paths = [s[0] for s in train_base.samples]
train_labels = [s[1] for s in train_base.samples]

val_paths = [s[0] for s in val_base.samples]
val_labels = [s[1] for s in val_base.samples]

test_paths = [s[0] for s in test_base.samples]
test_labels = [s[1] for s in test_base.samples]

class_names = train_base.classes
num_classes = len(class_names)

print("Classes:", class_names)
print("Train:", len(train_paths))
print("Val:", len(val_paths))
print("Test:", len(test_paths))

Classes: ['no_nodule', 'nodule']
Train: 4050
Val: 860
Test: 890


In [6]:
train_data = CTDataset(train_paths, train_labels, image_size=img_size)
val_data   = CTDataset(val_paths, val_labels, image_size=img_size)
test_data  = CTDataset(test_paths, test_labels, image_size=img_size)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=num_workers)

In [7]:
num_classes = 2
dropout_rate = 0.6

model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

model.fc = nn.Sequential(
    nn.Dropout(dropout_rate),
    nn.Linear(model.fc.in_features, num_classes)
)

model = model.to(device)
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [8]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

In [9]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    y_true, y_pred = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

        preds = out.argmax(1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

    loss = total_loss / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)
    return loss, acc

In [10]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            out = model(x)
            loss = criterion(out, y)

            total_loss += loss.item() * x.size(0)

            preds = out.argmax(1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    loss = total_loss / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)
    return loss, acc, y_true, y_pred

In [11]:
learning_rate = 1e-3
weight_decay = 1e-4

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=learning_rate,
    weight_decay=weight_decay
)

In [12]:
freeze_epochs = 5

best_acc = 0
best_wts = copy.deepcopy(model.state_dict())

for epoch in range(freeze_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

    print(f"Epoch {epoch+1}/{freeze_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        best_wts = copy.deepcopy(model.state_dict())

Epoch 1/5
Train Loss: 0.4855 | Train Acc: 0.7677
Val Loss:   0.4137 | Val Acc:   0.8453
Epoch 2/5
Train Loss: 0.4327 | Train Acc: 0.8012
Val Loss:   0.3880 | Val Acc:   0.8419
Epoch 3/5
Train Loss: 0.4235 | Train Acc: 0.8062
Val Loss:   0.3817 | Val Acc:   0.8384
Epoch 4/5
Train Loss: 0.4055 | Train Acc: 0.8119
Val Loss:   0.4138 | Val Acc:   0.8151
Epoch 5/5
Train Loss: 0.4051 | Train Acc: 0.8136
Val Loss:   0.3794 | Val Acc:   0.8256


In [13]:
model.load_state_dict(best_wts)

for param in model.parameters():
    param.requires_grad = True

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=weight_decay
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

In [14]:
epochs = 10

patience = 5
counter = 0

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

    scheduler.step(val_acc)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print(f"LR: {optimizer.param_groups[0]['lr']:.8f}")

    if val_acc >= best_acc:
        best_acc = val_acc
        best_wts = copy.deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered")
        break

Epoch 1/10
Train Loss: 0.3111 | Train Acc: 0.8642
Val Loss:   0.4717 | Val Acc:   0.8465
LR: 0.00010000
Epoch 2/10
Train Loss: 0.1094 | Train Acc: 0.9533
Val Loss:   0.5268 | Val Acc:   0.8233
LR: 0.00010000
Epoch 3/10
Train Loss: 0.0784 | Train Acc: 0.9669
Val Loss:   0.7165 | Val Acc:   0.8023
LR: 0.00010000
Epoch 4/10
Train Loss: 0.0522 | Train Acc: 0.9793
Val Loss:   0.7653 | Val Acc:   0.8140
LR: 0.00005000
Epoch 5/10
Train Loss: 0.0288 | Train Acc: 0.9881
Val Loss:   0.7161 | Val Acc:   0.8267
LR: 0.00005000
Epoch 6/10
Train Loss: 0.0296 | Train Acc: 0.9874
Val Loss:   0.8081 | Val Acc:   0.8233
LR: 0.00005000
Early stopping triggered


In [15]:
model.load_state_dict(best_wts)
print("Best Validation Accuracy:", best_acc)

Best Validation Accuracy: 0.8465116279069768


In [16]:
loss, acc, y_true, y_pred = evaluate(model, test_loader, criterion, device)

print("Test Accuracy:", acc)
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Benign", "Malignant"]))

Test Accuracy: 0.7674157303370787

Confusion Matrix:
[[ 70 180]
 [ 27 613]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.72      0.28      0.40       250
   Malignant       0.77      0.96      0.86       640

    accuracy                           0.77       890
   macro avg       0.75      0.62      0.63       890
weighted avg       0.76      0.77      0.73       890



In [18]:
torch.save(model.state_dict(), "resnet50_ct_best.pth")
print("Model saved")

Model saved


In [ ]:
loaded_model = models.resnet50(weights=None)
loaded_model.fc = nn.Sequential(
    nn.Dropout(dropout_rate),
    nn.Linear(loaded_model.fc.in_features, num_classes)
)

loaded_model.load_state_dict(torch.load("resnet50_ct_best.pth", map_location=device))
loaded_model = loaded_model.to(device)
loaded_model.eval()

print("Model loaded successfully")